# 02 — Base Model & Zero-Shot Baseline Evaluation
**Goal**: Load the small Code LLM (`deepseek-coder-1.3b-instruct`) using 4-bit quantization and LoRA adapters. Perform zero-shot generation to establish the Pass@1 baseline (Verification Check V1).

---

## Step 1: Environment Setup & Universal Path Resolution

In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

# Bypass Kaggle incompatible torchao version check in peft
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

print(f"Loading model and tokenizer: {MODEL_NAME} in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    load_in_4bit=False,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    attach_lora=True,
)

print("\n--- Trainable Parameters Summary ---")
if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()

# Save initial base checkpoint
os.makedirs("./checkpoints/base", exist_ok=True)
model.save_pretrained("./checkpoints/base")
tokenizer.save_pretrained("./checkpoints/base")
print("Saved base model checkpoint to ./checkpoints/base")

## Step 2: Load Model with 4-bit Quantization & LoRA Adapters
Uses BitsAndBytes 4-bit NF4 quantization to shrink memory usage from ~5GB to <1.5GB VRAM. Attaches LoRA adapters (`r=16, alpha=32`) to `q_proj` and `v_proj` layers.

In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

# Bypass Kaggle incompatible torchao version check in peft
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

print(f"Loading model and tokenizer: {MODEL_NAME} in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    load_in_4bit=False,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    attach_lora=True,
)

print("\n--- Trainable Parameters Summary ---")
if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()

# Save initial base checkpoint
os.makedirs("./checkpoints/base", exist_ok=True)
model.save_pretrained("./checkpoints/base")
tokenizer.save_pretrained("./checkpoints/base")
print("Saved base model checkpoint to ./checkpoints/base")

## Step 3: Zero-Shot Baseline Test (Verification Check V1)
Tests raw model code generation capabilities without debugging feedback ($K=1$, temperature=0.2).

In [ ]:
humaneval = load_dataset('openai_humaneval', split='test')
sample_problem = humaneval[0]['prompt']

print("--- HumanEval Problem 0 Prompt ---")
print(sample_problem[:250] + "...")

print("\nGenerating code solution...")
code = generate_code(model, tokenizer, sample_problem, temperature=0.2)
print("\n--- Generated Code ---")
print(code)

exec_result = run_code(code)
print(f"\nExecution Result: {exec_result}")

results = [{"status": exec_result["status"]}]
pass_at_1 = calculate_pass_at_1(results)
dist = calculate_error_distribution(results)
print(f"Baseline Pass@1: {pass_at_1 * 100:.1f}%")
print(f"Error Distribution: {dist}")
print("Verification Check V1 completed!")